### Non-additive PGS Demo Notebook - XGBoost

This notebook provides a minimal, standalone demonstration of how the XGBoost model was trained and evaluated for a single simulated phenotype in the paper Benchmarking non-additive genetic effects on polygenic prediction and machine learning-based approaches. It loads a demo dataset (one of the smaller synthetic phenotypes used in the manuscript), performs cross-validation using the same code structure as the full Snellius pipeline, runs Optuna-based hyperparameter tuning, and compares results to the published model outputs.

Because XGBoost uses multithreaded operations, models may differ slightly (<1×10⁻⁴ in R²) across runs even with the same random seed; this behavior is expected.

Phenotype:
- 100 causal SNPs
- total SNP heritability = 50%
- all causal SNPs have a dominance deviation ratio of k = -0.5

The total run time of the notebook (including loading + cleaning data and running the model) was < 40 minutes using a 2021 MacBook Pro (Sonoma 14.5; M1 Max; 32GB RAM).                     

#### DATA & CODE
GitHub + Readme: https://github.com/nybell/non-add-paper/tree/main         
Zenodo repo: https://zenodo.org/records/17552313                                                 
Manuscript DOI: https://doi.org/10.1101/2025.10.10.25337750                            
Questions: n.y.bell@vu.nl                   
______________________________________________________________________________

### Load required packages

The cell below loads all required packages used in the demo. Information for setting up the Python environment can be found at the GitHub link above. 

The custom functions ```adjusted_r2```, ```xgb_objective```, and ```split_data``` are imported from the ```model_definition.py```, which must be available in the same directory as the notebook (or on the Python path).

In [2]:
# load packages - basic
import math
import copy
import time
import pickle
import optuna
import random
import numpy as np
import pandas as pd
import xgboost as xgb
from tqdm import tqdm
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from model_definition import adjusted_r2, xgb_objective, split_data

In [18]:
# Set your desired seed for reproducibility
seed = 42

# Python's built-in random module
random.seed(seed)

### Load data & prep for model

The cell below specifies the file path for:
- the input data file containing the simulated phenotypes + causal SNPs
- the results file for the same phenotype as published (given here for comparison)

These file paths should be editted to match your own copies of the data. 

Note: If you want to run the code for any of the other simulated phenotypes, they can be found on the Zenodo repository (https://zenodo.org/records/17552313) in the ```sim_data_ml_ready_071125.tar.gz``` file. All model results can be found on the GitHub at ```non-add-paper/results/model_out/```

In [3]:
# set file path to data file
datafile = "/Users/nyb/demo_data/DATA_eur_nsnps100_h0.5_a0_d0.5_50k.txt"         # EDIT TO YOUR OWN FILE PATH

# set file path to published equivalent for same phenotype
pubfile = "/Users/nyb/demo_data/XGB_nsnps100_h0.5_a0_d0.5_50k_trials100.pkl"     # EDIT TO YOUR OWN FILE PATH

The cell below loads the data and performs some initial cleaning:
- remove ```IID``` from the beginning of IID numbers 
- ```IID``` here numbers are just fillers for the PLINK .fam file, since this is a simulated data set using fake individuals
- fill NAs with ```-1```
- replace ```"test"``` in ```split1``` with ```10```
- this is done to avoid errors later (split1 is not used for data splitting)
- ```data.head()``` checks data for sanity

In [ ]:
# load all data set
data = pd.read_csv(datafile, sep = "\t")
# remove "syn" from ids
data['IID'] = data['IID'].str.replace('syn', '').astype(int)
# fill NAs with 0.0
data = data.fillna(-1)
# replace "test" with "10" in split1 column and convert to int
data['split1'] = data['split1'].replace("test", "10").astype(int)
# check data
data.head()

,FID,IID,father,mother,sex,phenotype,split1,split2,additive.prs,domdev.add.comp,...,chr5:93258428:G:A_A,chr16:66071133:A:G_G,chr3:183188020:A:G_A,chr1:194841459:A:G_G,chr8:24862989:G:A_G,chr8:4779193:T:C_C,chr10:83170668:G:T_T,chr4:146028406:A:G_G,chr12:4856240:A:C_A,chr4:14365883:A:G_A
0,syn1,1,0,0,0,5.063255,1,5,5.316734,0,...,1,0,0,0,1,1,2,1,0,0
1,syn10,10,0,0,0,5.162452,2,4,6.075860,0,...,1,0,0,2,0,1,1,0,1,0
2,syn100,100,0,0,0,4.745832,3,2,6.302860,0,...,2,0,0,0,0,0,1,0,1,1
3,syn1000,1000,0,0,0,5.045312,2,5,5.456551,0,...,0,0,0,1,1,0,1,0,1,1
4,syn10000,10000,0,0,0,5.626505,4,1,6.434155,0,...,0,1,0,0,0,1,0,0,0,1


The cell below drops columns that are not used as predictors in the XGboost model:
- ```FID```, ```father```, ```mother```, ```sex``` and polygenic scores ```additve.prs```, ```domdev.add.comp```, ```domdev.dom.comp```,
               ```domdev.prs```
- ```IID``` is dropped later in the model runs (in the ```split_data``` function)
- a ```try/except``` block is used in case ```domdev.add.comp``` is missing (wasn't always added in when formatting final data sets)

In [22]:
# drop columns that are not needed
try:
    # Try to drop all the specified columns
    data.drop(['FID', 'father', 'mother', 'sex', 'additive.prs', 'domdev.add.comp', 'domdev.dom.comp',
               'domdev.prs'], axis=1, inplace=True)
except KeyError:
    # If there's an error, drop the same columns excluding 'domdev.add.comp'
    data.drop(['FID', 'father', 'mother', 'sex', 'additive.prs', 'domdev.dom.comp',
               'domdev.prs'], axis=1, inplace=True)


### Define hyperparams & settings

The cell belows sets a number of settings for the XGboost model run in the cell after:
- ```hyperparam_trials``` specifies the number of Optuna hyperparameter tuning trials used per CV split
- ```estimator_vals```, ```learning_rates```, and ```max_depth_vals``` define the ranges of values the objective fucntion will search (values are min and max)
- ```xgb_gpu``` is true or false depending on if there is a GPU. If you are running this on a machine with an NVIDIA GPU, then you can set this to true. At the time this code was written there was no native GPU support for Mac silicon via XGBoost. For the demo data, its fine to run on CPU. 
- ```test_data``` is initialized to record results.


In [23]:
# set hyper parameters
hyperparam_trials = 100
estimator_vals = [1000, 5000]
learning_rates = [0.01, 0.2]
max_depth_vals = [1]

# set gpu status
xgb_gpu = False  # EDIT TO YOUR OWN SETTING

# initialize test_data
test_data = {}

### Run model

Run XGBoost across all cross-validation splits.                     
For each split ((n = 5) using the ```split2``` to split):
- Partition data into training, validation, and test sets.
- Use Optuna to tune hyperparameters ((```xgb_objective```) using the train (```X_train```) and tune (```X_valid```) sets).
- Train a final model with early stopping.
- Evaluate validation and test performance (R²).
- Extract and store predictions and feature importances.
- All results are collected into a dictionary indexed by split number.

In [24]:
# --- time the entire cell ---
cell_start = time.time()

# start for loop
for split in range(1,6):

    # split data
    X_train, y_train, X_valid, y_valid, X_test, y_test, \
        ids_train, ids_tune, ids_test, tune_split = split_data(data, split_col = "split2",
                                                               split_num = split,
                                                               drop_col = "split1",
                                                               phenotype_col = "phenotype", 
                                                               fold_type = "cv")
    # print current split
    print("| ... CV split", tune_split, " ... |")

    # Use Optuna to find the best hyperparameters
    study = optuna.create_study(direction='maximize')
    print(f" ... Hyperparameter tuning ... \n")
    with tqdm(total=hyperparam_trials) as pbar:
        def wrapped_objective(trial):
            result = xgb_objective(trial, X_train, y_train, X_valid, y_valid, 
                                    estimators = estimator_vals, max_depth = max_depth_vals, 
                                    learning_rate = learning_rates, xgb_gpu = xgb_gpu)
            pbar.update(1)  # Update progress bar after each trial
            return result

    study.optimize(wrapped_objective, n_trials=hyperparam_trials)

    # Best parameters from Optuna
    best_params = study.best_params
    print(f"Best hyper-parameters: {best_params}")

    # check for gpu 
    if xgb_gpu:
        best_params.update({
            'tree_method': 'hist',   
            'device': 'cuda'})
        print(f"... Using GPU for final training ...")
    else:
        best_params.update({
            'tree_method': 'hist',})  
        print(f"... Using CPU for final training ...")
    # initialize
    print("... Initializing model ...")

    # Convert to DMatrix
    dtrain = xgb.DMatrix(X_train, label=y_train)
    dvalid = xgb.DMatrix(X_valid, label=y_valid)
    dtest = xgb.DMatrix(X_test, label=y_test)

    # Set fixed num_boost_round for final training
    final_boost_rounds = 10000  # or any large value

    # Train with early stopping
    booster = xgb.train(
        params=best_params,
        dtrain=dtrain,
        num_boost_round=final_boost_rounds,
        evals=[(dvalid, 'eval')],
        early_stopping_rounds=100,
        verbose_eval=False
    )

    # Predict on validation and test sets
    y_pred_tune = booster.predict(dvalid)
    y_pred_test = booster.predict(dtest)

    # Feature importance (example: gain-based)
    feat_importances = booster.get_score(importance_type='gain')

    # Evaluate the model on the test set
    tune_r2 = r2_score(y_valid, y_pred_tune)
    test_r2 = r2_score(y_test, y_pred_test)
    tune_adj_r2 = adjusted_r2(y_valid, y_pred_tune, len(y_valid), X_valid.shape[1])
    test_adj_r2 = adjusted_r2(y_test, y_pred_test, len(y_test), X_test.shape[1])
    print(f"XGBoost Tune Set: R2 = {tune_r2}")
    print(f"XGBoost Test Set: R2 = {test_r2}")

    # Record results
    split_key = f"split{tune_split}"
    test_dict = {"ids": ids_test, "phenotype": y_test, "r2": test_r2, "adj_r2": test_adj_r2, "feature_importances": feat_importances}
    test_data[split_key] = test_dict

print("| ---- Finished XGBoost model training ---- |", "\n")

# --- print total cell time ---
cell_elapsed = time.time() - cell_start
print(f"| ---- Finished XGBoost model training in {cell_elapsed/60:.2f} minutes ---- |", "\n")


[I 2025-11-15 13:32:14,383] A new study created in memory with name: no-name-049c14b0-4caa-486b-90a2-0ec2291c14fd


| ... CV split 1  ... |
 ... Hyperparameter tuning ... 



  0%|          | 0/100 [00:00<?, ?it/s]

XGBoost training parameters: {'max_depth': 1, 'learning_rate': 0.18115885708411214, 'min_child_weight': 3, 'colsample_bytree': 0.9072289952458678, 'reg_alpha': 0.21603213391776133, 'reg_lambda': 1.2057858603439142, 'objective': 'reg:squarederror', 'eval_metric': 'rmse', 'random_state': 707, 'tree_method': 'hist'}
 ... xgb_gpu =  False  ... 
... Initializing model ...



[I 2025-11-15 13:32:17,905] Trial 0 finished with value: 0.4897161523188066 and parameters: {'max_depth': 1, 'learning_rate': 0.18115885708411214, 'min_child_weight': 3, 'colsample_bytree': 0.9072289952458678, 'reg_alpha': 0.21603213391776133, 'reg_lambda': 1.2057858603439142, 'n_estimators': 3188}. Best is trial 0 with value: 0.4897161523188066.
[I 2025-11-15 13:32:24,066] Trial 1 finished with value: 0.48957225453336506 and parameters: {'max_depth': 1, 'learning_rate': 0.10627435906189399, 'min_child_weight': 5, 'colsample_bytree': 0.6614066343375332, 'reg_alpha': 0.8441417048370539, 'reg_lambda': 0.7430555886437262, 'n_estimators': 4374}. Best is trial 0 with value: 0.4897161523188066.
[I 2025-11-15 13:32:28,364] Trial 2 finished with value: 0.4899645739793078 and parameters: {'max_depth': 1, 'learning_rate': 0.18895932755177944, 'min_child_weight': 4, 'colsample_bytree': 0.7282353001038987, 'reg_alpha': 0.11638888632683886, 'reg_lambda': 8.855919290026867, 'n_estimators': 4180}. B

...

XGBoost Tune Set: R2 = 0.5042424830088145
XGBoost Test Set: R2 = 0.4926787656869265
| ... CV split 5  ... |
 ... Hyperparameter tuning ... 



  0%|          | 0/100 [00:00<?, ?it/s]

XGBoost training parameters: {'max_depth': 1, 'learning_rate': 0.14405918051678657, 'min_child_weight': 6, 'colsample_bytree': 0.7822584567530769, 'reg_alpha': 0.8419581248699844, 'reg_lambda': 8.740462142980249, 'objective': 'reg:squarederror', 'eval_metric': 'rmse', 'random_state': 707, 'tree_method': 'hist'}
 ... xgb_gpu =  False  ... 
... Initializing model ...



[I 2025-11-15 14:00:56,408] Trial 0 finished with value: 0.4922904472435491 and parameters: {'max_depth': 1, 'learning_rate': 0.14405918051678657, 'min_child_weight': 6, 'colsample_bytree': 0.7822584567530769, 'reg_alpha': 0.8419581248699844, 'reg_lambda': 8.740462142980249, 'n_estimators': 1763}. Best is trial 0 with value: 0.4922904472435491.
[I 2025-11-15 14:01:00,594] Trial 1 finished with value: 0.49773541530390986 and parameters: {'max_depth': 1, 'learning_rate': 0.12927430003786125, 'min_child_weight': 6, 'colsample_bytree': 0.6125359452298055, 'reg_alpha': 0.8645731698451004, 'reg_lambda': 4.748105636279969, 'n_estimators': 2708}. Best is trial 1 with value: 0.49773541530390986.
[I 2025-11-15 14:01:05,823] Trial 2 finished with value: 0.4478446735499957 and parameters: {'max_depth': 1, 'learning_rate': 0.038317956190243106, 'min_child_weight': 3, 'colsample_bytree': 0.8108798826248337, 'reg_alpha': 0.49008590185816214, 'reg_lambda': 3.0388820346030454, 'n_estimators': 3357}. B

Best hyper-parameters: {'max_depth': 1, 'learning_rate': 0.16297772212425976, 'min_child_weight': 4, 'colsample_bytree': 0.8566336658032633, 'reg_alpha': 0.8310103419561126, 'reg_lambda': 1.8936551554626688, 'n_estimators': 4540}
... Using CPU for final training ...
... Initializing model ...
XGBoost Tune Set: R2 = 0.4994612694674483
XGBoost Test Set: R2 = 0.5036494694658273
| ---- Finished XGBoost model training ---- | 

| ---- Finished XGBoost model training in 35.35 minutes ---- | 



In [ ]:
# FORMAT THE TEST DATA RESULTS
# convert results to df
test_df = pd.DataFrame(test_data).transpose()

### Check model performance & compare with published result

The cell below prints the mean and standard deviation of the variance-explained across the test splits.  

In [29]:
# check mean r2 across test splits
print("Mean test R2:", test_df['r2'].mean())

# check sd r2 across test splits
print("SD test R2:", test_df['r2'].std())

Mean test R2: 0.49665157808895255
SD test R2: 0.005257097543871927


In the cells below we:
- load the pickle file containined the original (published) results for the same phenotype
- and print the mean and SD of the variance explained in the test set

These results should be *highly* consistent with the local run - with variation seen here < 0.0001. This variation is expected, as software environments are slightly different between the local (this notebook) and the HPC (published) versions      

(see documentation: https://xgboost.readthedocs.io/en/stable/tutorials/learning_to_rank.html?utm_source=chatgpt.com#reproducible-result)

In [ ]:
# load published results for comparison
with open(pubfile, "rb") as f:
    published_results = pickle.load(f)


In [37]:
# check mean r2 across test splits
print("Mean test R2:", published_results['r2'].mean())

# check sd r2 across test splits
print("SD test R2:", published_results['r2'].std())

Mean test R2: 0.4966310475883292
SD test R2: 0.005249069771997958
